# Notebook 4: Correlation & Feature Selection
Stages 17–20

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
df=pd.read_excel("customer_data_raw.xlsx").drop_duplicates()

## Stage 17 - Correlation Analysis

In [ ]:
num=df.select_dtypes(include='number')
corr=num.corr()
corr.head()

In [ ]:
high=[]
cols=corr.columns
for i in range(len(cols)):
    for j in range(i):
        if abs(corr.iloc[i,j])>0.9:
            high.append((cols[i],cols[j],corr.iloc[i,j]))
high

In [ ]:
to_drop=['Salary_Copy','Correlation2','Age_Copy']
df_corr=df.drop(columns=[c for c in to_drop if c in df.columns])
df_corr.shape

## Stage 18 - Remove Constant Features

In [ ]:
df.nunique().sort_values().head(10)

In [ ]:
selector=VarianceThreshold(threshold=0)
X=df.select_dtypes(include='number').fillna(0)
X_new=selector.fit_transform(X)
selected=X.columns[selector.get_support()]
selected

## Stage 19 - Detect Data Leakage

In [ ]:
df[['LeakageFeature','Target']].head()

In [ ]:
df=df.drop(columns=['LeakageFeature'])
df.columns

## Stage 20 - Feature Selection

In [ ]:
X=df.select_dtypes(include='number').drop(columns=['Target']).fillna(0)
y=df['Target']
skb=SelectKBest(score_func=f_classif,k=5)
Xk=skb.fit_transform(X,y)
selected_features=X.columns[skb.get_support()]
selected_features

In [ ]:
model=RandomForestClassifier(random_state=42)
model.fit(X,y)
imp=pd.Series(model.feature_importances_,index=X.columns).sort_values(ascending=False)
imp.head(10)

In [ ]:
rfe=RFE(RandomForestClassifier(random_state=42),n_features_to_select=5)
rfe.fit(X,y)
X.columns[rfe.support_]

## Practice

1. Find all features with correlation >0.8.
2. Remove ConstantCol manually.
3. Try SelectKBest with k=10.
4. Compare RFE and Random Forest selected features.
